In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader , TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

d:\Machine Learing\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

In [3]:
def ingest_file(file_path , persist_dir = "chroma_db"):
    if file_path.endswith(".pdf"):
        loader = PyPDFLoader(file_path)
    else:
        loader = TextLoader(file_path)
    documents = loader.load()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = 800,
        chunk_overlap = 100
    )
    chunks = splitter.split_documents(documents)

    embedding = HuggingFaceEmbeddings(
        model_name = EMBEDDING_MODEL
    )
    db = Chroma.from_documents(
        chunks,
        embedding,
        persist_directory=persist_dir
    )
    return db

In [4]:
ingest_file("../data/docs.txt")

RAG Logic

In [5]:
from langchain_ollama import OllamaLLM
from langchain_chroma import Chroma
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

In [6]:
def get_rag_chain(persist_dir = "chroma_db"):
    embeddings = HuggingFaceEmbeddings(
        model_name = EMBEDDING_MODEL
    )
    db = Chroma(
        persist_directory= persist_dir,
        embedding_function=embeddings
    )
    retriever = db.as_retriever(search_kwargs={"k":3})
    llm = OllamaLLM(
        model = "llama3",
        temperature = 0,
        
    )
    qa = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
         verbose=False
    )
    return qa

In [7]:
qa = get_rag_chain()
response = qa.invoke({"query": "What is this document about?"})
print(response["result"])

This document appears to be about a technique called Retrieval-Augmented Generation (RAG) that enhances large language models by retrieving relevant external documents and providing them as context during response generation.


In [8]:
import streamlit as st

In [10]:
UPLOAD_DIR = "data/uploaded_files"
os.makedirs(UPLOAD_DIR,exist_ok=True)

st.set_page_config(page_title="RAG AI Assistant",layout="wide")

st.title ("RAG AI Assistant")
st.write("PDF QA")

mode = st.sidebar.selectbox(
    "Select Mode",
    ["PDF Question Answering"]
)
uploaded_file = st.file_uploader(
    "Upload PDF or Text file",
    type=["pdf","txt"]
)
if uploaded_file:
    file_path = os.path.join(UPLOAD_DIR,uploaded_file.name)
    with open(file_path,"wb") as f:
        f.write(uploaded_file.read())
    
    if st.button("ingest document"):
        with st.spinner("Ingesting document"):
            ingest_file(file_path)
        st.success("Document ingested success")
st.divider()

query = st.text_input("Ask your question")

if st.button("Get answer"):
    if not query:
        st.warning("please enter a question")
    else:
        with st.spinner("Thinking..."):
            qa_chain = get_rag_chain()
            if mode == "CV Analyzer":
                query = f"""Analyze the resume and answer:
                1. Strengths
                2. Weakness
                3. Missing skills 
                Resume Question: {query}
            """
            response = qa_chain.invoke({"query": query})
            st.subheader("Answer")
            st.write(response["result"])

2026-02-08 15:24:14.754 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 15:24:14.758 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 15:24:14.760 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 15:24:14.761 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 15:24:14.764 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 15:24:14.765 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 15:24:14.766 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 15:24:14.768 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar